# 03. Workflow AI w Colabie: router, klasyfikacja i mała sieć neuronowa

Ten notebook pokazuje, jak projektować workflow wokół LLM:

- kiedy wystarczy prompt,
- kiedy potrzebny jest RAG,
- kiedy trzeba użyć narzędzia,
- kiedy sens ma trening lub fine-tuning.

Zawiera wersję regułową, wersję z małą siecią neuronową oraz opcjonalną wersję z lokalnym LLM-em.

In [1]:
!pip -q install scikit-learn pandas transformers accelerate sentencepiece


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch

print("GPU dostępne:", torch.cuda.is_available())

GPU dostępne: True


## 1. Regułowy router workflow

To najprostsza wersja systemu decyzyjnego.

In [3]:
def workflow_router(question):
    q = question.lower()

    document_words = ["dokument", "pdf", "regulamin", "instrukcja", "procedura", "sylabus", "umowa"]
    calculation_words = ["policz", "oblicz", "średnia", "suma", "wykres", "csv", "excel", "tabela"]
    training_words = ["trening", "trenować", "dostroić", "fine-tuning", "nauczyć model", "mój styl"]

    if any(word in q for word in document_words):
        return "RAG", "Pytanie dotyczy dokumentów lub bazy wiedzy."
    elif any(word in q for word in calculation_words):
        return "TOOL", "Pytanie wymaga obliczeń albo pracy na danych."
    elif any(word in q for word in training_words):
        return "TRAINING", "Pytanie dotyczy uczenia lub dostrajania modelu."
    else:
        return "PROMPT", "Wystarczy zwykła odpowiedź modelu."

cases = [
    "Wyjaśnij mi, czym jest LLM.",
    "Mam dokument PDF i chcę pytać o jego treść.",
    "Policz średnią ocen z pliku CSV.",
    "Chcę dostroić model do mojego stylu pisania.",
    "Czy regulamin pozwala oddać projekt po terminie?"
]

for case in cases:
    category, reason = workflow_router(case)
    print("=" * 90)
    print("PYTANIE:", case)
    print("KATEGORIA:", category)
    print("UZASADNIENIE:", reason)

PYTANIE: Wyjaśnij mi, czym jest LLM.
KATEGORIA: PROMPT
UZASADNIENIE: Wystarczy zwykła odpowiedź modelu.
PYTANIE: Mam dokument PDF i chcę pytać o jego treść.
KATEGORIA: RAG
UZASADNIENIE: Pytanie dotyczy dokumentów lub bazy wiedzy.
PYTANIE: Policz średnią ocen z pliku CSV.
KATEGORIA: TOOL
UZASADNIENIE: Pytanie wymaga obliczeń albo pracy na danych.
PYTANIE: Chcę dostroić model do mojego stylu pisania.
KATEGORIA: TRAINING
UZASADNIENIE: Pytanie dotyczy uczenia lub dostrajania modelu.
PYTANIE: Czy regulamin pozwala oddać projekt po terminie?
KATEGORIA: RAG
UZASADNIENIE: Pytanie dotyczy dokumentów lub bazy wiedzy.


## 2. Dane treningowe dla małej sieci neuronowej

Tu uczymy klasyfikator, który przewiduje ścieżkę workflow.

In [4]:
texts = [
    "Wyjaśnij czym jest sztuczna inteligencja",
    "Napisz krótką definicję LLM",
    "Co to jest transformer",
    "Wyjaśnij embedding prostym językiem",

    "Mam dokument PDF i chcę zadawać pytania",
    "Odpowiedz na podstawie regulaminu",
    "Mam bazę wiedzy i chcę z niej korzystać",
    "Chcę pytać o treść instrukcji BHP",

    "Policz średnią ocen studentów",
    "Wykonaj analizę danych z pliku CSV",
    "Oblicz odchylenie standardowe",
    "Zrób wykres sprzedaży miesięcznej",

    "Chcę, żeby model odpowiadał zawsze w stylu prawniczym",
    "Chcę dostroić model do mojego tonu wypowiedzi",
    "Chcę nauczyć model klasyfikować moje teksty",
    "Potrzebuję trenować model na przykładach"
]

labels = [
    "PROMPT", "PROMPT", "PROMPT", "PROMPT",
    "RAG", "RAG", "RAG", "RAG",
    "TOOL", "TOOL", "TOOL", "TOOL",
    "TRAINING", "TRAINING", "TRAINING", "TRAINING"
]

pd.DataFrame({"text": texts, "label": labels})

,text,label
0,Wyjaśnij czym jest sztuczna inteligencja,PROMPT
1,Napisz krótką definicję LLM,PROMPT
2,Co to jest transformer,PROMPT
3,Wyjaśnij embedding prostym językiem,PROMPT
4,Mam dokument PDF i chcę zadawać pytania,RAG
5,Odpowiedz na podstawie regulaminu,RAG
6,Mam bazę wiedzy i chcę z niej korzystać,RAG
7,Chcę pytać o treść instrukcji BHP,RAG
8,Policz średnią ocen studentów,TOOL
9,Wykonaj analizę danych z pliku CSV,TOOL


## 3. Trening małej sieci neuronowej `MLPClassifier`

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier

classifier = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        max_iter=1000,
        random_state=42
    ))
])

classifier.fit(texts, labels)
print("Model wytrenowany.")

Model wytrenowany.


In [6]:
test_questions = [
    "Mam dokumenty uczelni i chcę zadawać pytania o zasady zaliczenia",
    "Policz sumę sprzedaży z tabeli",
    "Wyjaśnij, czym jest RAG",
    "Chcę nauczyć model rozpoznawania mojego stylu",
    "Czy na podstawie regulaminu mogę mieć trzy nieobecności?"
]

predictions = classifier.predict(test_questions)

for question, prediction in zip(test_questions, predictions):
    print(question, "=>", prediction)

Mam dokumenty uczelni i chcę zadawać pytania o zasady zaliczenia => RAG
Policz sumę sprzedaży z tabeli => TOOL
Wyjaśnij, czym jest RAG => PROMPT
Chcę nauczyć model rozpoznawania mojego stylu => TRAINING
Czy na podstawie regulaminu mogę mieć trzy nieobecności? => RAG


## 4. Porównanie: router regułowy vs sieć neuronowa

In [7]:
comparison = []

for question in test_questions:
    rule_category, rule_reason = workflow_router(question)
    ml_category = classifier.predict([question])[0]
    comparison.append({
        "question": question,
        "rule_router": rule_category,
        "mlp_classifier": ml_category
    })

pd.DataFrame(comparison)

,question,rule_router,mlp_classifier
0,Mam dokumenty uczelni i chcę zadawać pytania o...,RAG,RAG
1,Policz sumę sprzedaży z tabeli,TOOL,TOOL
2,"Wyjaśnij, czym jest RAG",PROMPT,PROMPT
3,Chcę nauczyć model rozpoznawania mojego stylu,TRAINING,TRAINING
4,Czy na podstawie regulaminu mogę mieć trzy nie...,RAG,RAG


## 5. Opcjonalnie: router oparty o lokalny LLM

Ta część pobiera mały model Qwen i używa go jako klasyfikatora intencji.

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Urządzenie:", device)

# Uwaga: na GPU używamy float16, na CPU float32.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
model.to(device)
model.eval()

print("Model załadowany:", MODEL_NAME)

/home/jakub/PycharmProjects/SWPS_2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Urządzenie: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1258.59it/s]


Model załadowany: Qwen/Qwen2.5-0.5B-Instruct


In [9]:
def ask_llm(prompt, system="Jesteś pomocnym asystentem dydaktycznym. Odpowiadasz po polsku, krótko i precyzyjnie.", max_new_tokens=250):
    """Prosta funkcja do rozmowy z lokalnym modelem uruchomionym w Colabie."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    return answer.strip()

In [10]:
def classify_intent_with_llm(user_question):
    prompt = f"""
Zaklasyfikuj intencję użytkownika do jednej z kategorii:

PROMPT_ONLY - wystarczy zwykłe pytanie do modelu
RAG - potrzebne są dokumenty lub baza wiedzy
TRAINING - potrzebny jest trening albo dostrajanie modelu
TOOL - potrzebne jest narzędzie zewnętrzne, np. Python, kalkulator, baza danych

Pytanie użytkownika:
{user_question}

Odpowiedz dokładnie w takim formacie:
KATEGORIA: ...
UZASADNIENIE: ...
"""
    return ask_llm(prompt, max_new_tokens=150)

for q in test_questions:
    print("=" * 90)
    print("PYTANIE:", q)
    print(classify_intent_with_llm(q))

PYTANIE: Mam dokumenty uczelni i chcę zadawać pytania o zasady zaliczenia
KATEGORIA: TOOL
UZASADNIENIE: Dokumenty uczelni
PYTANIE: Policz sumę sprzedaży z tabeli
KATEGORIA: TOOL
UZASADNIENIE: Prawdopodobieństwo sprzedaży
Rozwiązanie: Zapytanie "Policz sumę sprzedaży z tabeli" jest odpowiedzią na to pytanie.
PYTANIE: Wyjaśnij, czym jest RAG
KATEGORIA: TOOL
UZASADNIENIE: RAG
PYTANIE: Chcę nauczyć model rozpoznawania mojego stylu
KATEGORIA: TOOL
UZASADNIENIE: Rozpoznawanie stylu
PYTANIE: Czy na podstawie regulaminu mogę mieć trzy nieobecności?
KATEGORIA: PROMPT_ONLY
UZASADNIENIE: Czy na podstawie regulaminu mogę mieć trzy nieobecności?


## Puenta dydaktyczna

LLM jest tylko jednym elementem systemu. W praktyce potrzebujemy routerów, klasyfikatorów, reguł, narzędzi, baz wiedzy i procedur kontroli jakości.